# ⚡ EfficientNet — Notes + Interview
---
> **Simple English** | **Interview Ready** | Year: 2019 | Creator: Google Brain (Tan & Le)

## 📌 What is EfficientNet? (Simple English)
- EfficientNet asked: what's the **best way to scale up a CNN?**
- Three ways to scale: make it **wider** (more channels), **deeper** (more layers), or use **higher resolution** input
- EfficientNet scales **all three together** using a compound coefficient **φ**
- Result: much better accuracy with **fewer parameters** than all previous models
- EfficientNet-B7 achieved SOTA on ImageNet with just 66M params

## 🔑 Compound Scaling
```
depth    = α^φ
width    = β^φ
resolution = γ^φ

Where α, β, γ are found by grid search
φ = compound coefficient (B0→φ=0, B7→φ=2.31)
```

## 🧱 EfficientNet Variants
| Model | Params | ImageNet Top-1 |
|---|---|---|
| EfficientNet-B0 | 5.3M | 77.1% |
| EfficientNet-B1 | 7.8M | 79.1% |
| EfficientNet-B4 | 19M | 82.9% |
| EfficientNet-B7 | 66M | 84.3% |
| ResNet-50 | 25M | 76.0% |
| VGG16 | 138M | 71.3% |

In [ ]:
import tensorflow as tf
import numpy as np

# EfficientNetB0 — smallest and fastest variant
eff_b0 = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
print(f"EfficientNetB0:")
print(f"  Layers : {len(eff_b0.layers)}")
print(f"  Params : {eff_b0.count_params():,}  (~4M)")

print("\nComparison:")
models_info = {
    'VGG16'          : (tf.keras.applications.VGG16, (224,224,3)),
    'ResNet50'       : (tf.keras.applications.ResNet50, (224,224,3)),
    'EfficientNetB0' : (tf.keras.applications.EfficientNetB0, (224,224,3)),
    'EfficientNetB4' : (tf.keras.applications.EfficientNetB4, (380,380,3)),
}
for name,(ModelClass, ishape) in models_info.items():
    m = ModelClass(weights=None,include_top=False,input_shape=ishape)
    print(f"  {name:20s}: {m.count_params():>10,} params")

In [ ]:
# Transfer Learning with EfficientNetB0 — most practical choice
eff_b0 = tf.keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(224,224,3)
)

# Fine-tuning: freeze most layers, unfreeze last few
eff_b0.trainable = False
for layer in eff_b0.layers[-20:]:   # unfreeze last 20 layers
    layer.trainable = True

x = eff_b0.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(10, activation='softmax')(x)

model_eff = tf.keras.Model(eff_b0.input, output)
model_eff.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

trainable = sum([tf.size(w).numpy() for w in model_eff.trainable_weights])
total     = sum([tf.size(w).numpy() for w in model_eff.weights])
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print("\nEfficientNetB0 is the best model for most practical tasks!")
print("Best accuracy/parameter ratio of any CNN architecture.")

## 🗣️ Interview Q&A

**Q: What is EfficientNet?**
> A family of CNNs that scale width, depth, and resolution simultaneously using a compound coefficient. Achieves state-of-the-art accuracy with much fewer parameters than previous models.

**Q: What is compound scaling?**
> Instead of scaling only one dimension (deeper OR wider OR higher-res), compound scaling scales all three in a balanced way. This avoids diminishing returns from scaling just one dimension.

**Q: Which EfficientNet to use in practice?**
> B0 for fast/mobile (5M params), B4 for good balance (19M), B7 for max accuracy (66M). For production, B0 or B1 is most common. Use with pretrained ImageNet weights.

**Q: EfficientNet vs ResNet for transfer learning?**
> EfficientNet usually achieves better accuracy with fewer parameters. EfficientNetB0 (5M) > ResNet50 (25M) in most tasks. EfficientNet is the go-to choice for new projects.

## 📋 Architecture Timeline Summary
| Year | Model | Params | Top Innovation |
|---|---|---|---|
| 1998 | LeNet | 60K | First CNN ✅ |
| 2012 | AlexNet | 60M | ReLU + Dropout + GPU |
| 2014 | VGG16 | 138M | 3×3 filters, depth |
| 2014 | Inception | 6.7M | Parallel filters, 1×1 bottleneck |
| 2015 | ResNet | 25M | Skip connections |
| 2019 | EfficientNet | 5M | Compound scaling ✅ |